[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/rl-textbook/blob/main/notebooks/ch10_reward_models.ipynb)

<div style="background: linear-gradient(135deg, #1B474D 0%, #20808D 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">Chapter 10 — Reward Shaping & Reward Models</h1>
  <p style="color: #BCE2E7; margin-top: 10px; font-size: 1.1em;">
    How to design and learn reward signals for RL from Human Feedback (RLHF).
    We cover potential-based shaping, learned reward models, Bradley-Terry preferences,
    and the reward over-optimisation problem.
  </p>
</div>

**Learning objectives:**
- Understand potential-based reward shaping and why it preserves optimal policies
- Use a HuggingFace sentiment classifier as a proxy reward model
- Implement Bradley-Terry preference ranking from synthetic data
- Observe reward over-optimisation (Goodhart's Law in RL)

<div style="background: #1C1B19; border-left: 4px solid #20808D; padding: 15px; border-radius: 6px; margin: 10px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">Environment Setup</h2>
  <p style="color: #CDCCCA; margin-top: 8px;">Run this notebook on Google Colab (free T4 GPU) or Kaggle. CPU is sufficient for most experiments; GPU speeds up the sentiment model. Expected total runtime: ~8 minutes.</p>
</div>

In [ ]:
!pip install -q transformers accelerate datasets trl torch matplotlib numpy tqdm

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import pipeline
from scipy.special import expit as sigmoid
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#F7F6F2',
    'axes.facecolor': '#F9F8F5',
    'axes.edgecolor': '#D4D1CA',
    'axes.labelcolor': '#28251D',
    'text.color': '#28251D',
    'xtick.color': '#7A7974',
    'ytick.color': '#7A7974',
    'grid.color': '#D4D1CA',
    'font.family': 'sans-serif',
})
print('Setup complete.')

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§10.1 Potential-Based Reward Shaping</h2>
</div>

### Theory

In many sparse-reward environments an agent receives zero reward for almost every step, making learning very slow. **Reward shaping** adds an auxiliary reward to guide exploration without changing the optimal policy.

**Potential-based shaping (Ng et al., 1999)** is the gold standard: define a *potential function* \(\Phi(s)\) and add the shaping reward

\[ F(s, s') = \gamma \Phi(s') - \Phi(s) \]

to every transition. The shaped MDP has **the same optimal policy** as the original, because \(F\) is a telescoping sum that cancels over any complete trajectory.

Below we build a 5×5 GridWorld with a goal in the top-right corner and verify this invariance.

In [ ]:
# ── GridWorld ──────────────────────────────────────────────────────────────
class GridWorld:
    """5x5 GridWorld. Goal at (4,4). Walls at specified cells."""
    def __init__(self, size=5, goal=(4, 4), gamma=0.95):
        self.size = size
        self.goal = goal
        self.gamma = gamma
        self.n_states = size * size
        self.n_actions = 4  # 0=up, 1=down, 2=left, 3=right
        self.walls = {(1,1), (1,2), (2,2), (3,2)}

    def state_to_xy(self, s):
        return (s // self.size, s % self.size)

    def xy_to_state(self, r, c):
        return r * self.size + c

    def step(self, s, a):
        r, c = self.state_to_xy(s)
        dr, dc = [(-1,0),(1,0),(0,-1),(0,1)][a]
        nr, nc = np.clip(r+dr, 0, self.size-1), np.clip(c+dc, 0, self.size-1)
        if (nr, nc) in self.walls:
            nr, nc = r, c
        ns = self.xy_to_state(nr, nc)
        done = (nr, nc) == self.goal
        reward = 1.0 if done else 0.0
        return ns, reward, done

    def potential(self, s):
        """Manhattan-distance potential: closer to goal → higher value."""
        r, c = self.state_to_xy(s)
        gr, gc = self.goal
        dist = abs(r - gr) + abs(c - gc)
        return -dist  # negative distance: 0 at goal

    def shaping_reward(self, s, ns):
        return self.gamma * self.potential(ns) - self.potential(s)


def value_iteration(env, shaped=False, tol=1e-6, max_iter=1000):
    """Tabular value iteration. Returns V and greedy policy."""
    V = np.zeros(env.n_states)
    goal_state = env.xy_to_state(*env.goal)
    for _ in range(max_iter):
        V_new = np.zeros_like(V)
        for s in range(env.n_states):
            if s == goal_state:
                V_new[s] = 0.0
                continue
            q_vals = []
            for a in range(env.n_actions):
                ns, r, done = env.step(s, a)
                if shaped:
                    r += env.shaping_reward(s, ns)
                q = r + (0.0 if done else env.gamma * V[ns])
                q_vals.append(q)
            V_new[s] = max(q_vals)
        if np.max(np.abs(V_new - V)) < tol:
            break
        V = V_new
    policy = np.zeros(env.n_states, dtype=int)
    for s in range(env.n_states):
        q_vals = []
        for a in range(env.n_actions):
            ns, r, done = env.step(s, a)
            if shaped:
                r += env.shaping_reward(s, ns)
            q = r + (0.0 if done else env.gamma * V[ns])
            q_vals.append(q)
        policy[s] = np.argmax(q_vals)
    return V, policy


env = GridWorld()
V_orig, pi_orig = value_iteration(env, shaped=False)
V_shaped, pi_shaped = value_iteration(env, shaped=True)

print('Policies identical:', np.array_equal(pi_orig, pi_shaped))
print('Max |ΔV|:', np.max(np.abs(V_orig - V_shaped)).round(4))

In [ ]:
# ── Visualise GridWorld policies ──────────────────────────────────────────
arrows = ['↑', '↓', '←', '→']
colors = ['#20808D', '#A84B2F', '#1B474D', '#BCE2E7']

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax_idx, (ax, V, pi, title) in enumerate(zip(
    axes,
    [V_orig, V_shaped, V_orig],
    [pi_orig, pi_shaped, pi_orig],
    ['Original Reward — Value Function', 'Shaped Reward — Value Function',
     'Policy (identical in both)'])):

    grid = V.reshape(env.size, env.size) if ax_idx < 2 else np.zeros((env.size, env.size))
    im = ax.imshow(grid, cmap='Blues', origin='upper')

    for s in range(env.n_states):
        r, c = env.state_to_xy(s)
        if (r, c) == env.goal:
            ax.text(c, r, '★', ha='center', va='center', fontsize=18, color='#FFC553')
        elif (r, c) in env.walls:
            ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='#28251D'))
        else:
            label = f'{arrows[pi[s]]}\n{V[s]:.2f}' if ax_idx < 2 else arrows[pi[s]]
            ax.text(c, r, label, ha='center', va='center',
                    fontsize=11 if ax_idx < 2 else 18, color='#28251D')

    ax.set_title(title, fontsize=11, fontweight='bold', color='#28251D')
    ax.set_xticks([]); ax.set_yticks([])
    if ax_idx < 2:
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Potential-Based Shaping: Same Optimal Policy, Different Value Scales',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ch10_fig1_shaping.png', dpi=120, bbox_inches='tight')
plt.show()
print('Fig 1: Policy arrows identical — shaping preserves the optimal policy.')

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§10.2 Reward Model via Sentiment Classifier</h2>
</div>

In RLHF we do not have access to the true human reward — we must **learn it** from preference data. A common proxy is a fine-tuned classifier that predicts human ratings.

Here we use `distilbert-base-uncased-finetuned-sst-2-english` from HuggingFace as a stand-in *reward model*: the probability assigned to class **POSITIVE** becomes the scalar reward signal \(r_\phi(x, y) \in [0, 1]\).

In [ ]:
# ~1 min on CPU, ~15 s on GPU
device = 0 if torch.cuda.is_available() else -1
sentiment_rm = pipeline(
    'text-classification',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    device=device
)

# Candidate responses of varying quality
responses = [
    "This is an absolutely fantastic and wonderful explanation!",
    "This explanation is quite good and helpful.",
    "The explanation is okay, I suppose.",
    "This explanation is confusing and hard to follow.",
    "This is a terrible and useless response, completely wrong.",
]

def get_reward(text):
    result = sentiment_rm(text)[0]
    if result['label'] == 'POSITIVE':
        return result['score']
    else:
        return 1.0 - result['score']

rewards = [get_reward(r) for r in responses]
for resp, rew in zip(responses, rewards):
    print(f'  r={rew:.3f} | {resp[:60]}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
short_labels = [r[:40] + '...' for r in responses]
bar_colors = ['#20808D' if r > 0.6 else '#A84B2F' if r < 0.4 else '#FFC553' for r in rewards]
bars = ax.barh(range(len(rewards)), rewards, color=bar_colors, edgecolor='white')
ax.set_yticks(range(len(rewards)))
ax.set_yticklabels(short_labels, fontsize=9)
ax.set_xlabel('Reward Model Score (P(POSITIVE))')
ax.set_xlim(0, 1)
ax.axvline(0.5, color='#7A7974', linestyle='--', linewidth=1, label='Decision threshold')
ax.set_title('§10.2 — Sentiment Classifier as a Reward Model', fontweight='bold')
ax.legend()
for i, (bar, val) in enumerate(zip(bars, rewards)):
    ax.text(val + 0.01, i, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('ch10_fig2_reward_model.png', dpi=120, bbox_inches='tight')
plt.show()

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§10.3 Bradley-Terry Preference Ranking</h2>
</div>

The **Bradley-Terry model** is the workhorse of preference learning in RLHF. Given a pair of responses \((y_w, y_l)\) where \(y_w\) is preferred, we model:

\[ P(y_w \succ y_l) = \sigma(r_\phi(x, y_w) - r_\phi(x, y_l)) \]

and maximise the log-likelihood over observed preference pairs:

\[ \mathcal{L}_{\text{BT}} = -\mathbb{E}_{(y_w,y_l) \sim \mathcal{D}}\bigl[\log \sigma(r_\phi(y_w) - r_\phi(y_l))\bigr] \]

Here we simulate this with synthetic scalar rewards and recover the latent rankings.

In [ ]:
np.random.seed(0)

# --- Synthetic setup ---
N_ITEMS = 6
true_scores = np.array([3.0, 1.5, 2.5, 0.5, 2.0, 1.0])  # latent quality

# Generate pairwise preference data (Bradley-Terry sampling)
pairs = []
for i in range(N_ITEMS):
    for j in range(i+1, N_ITEMS):
        p_ij = sigmoid(true_scores[i] - true_scores[j])
        winner = i if np.random.rand() < p_ij else j
        loser = j if winner == i else i
        pairs.append((winner, loser))

print(f'Generated {len(pairs)} preference pairs from {N_ITEMS} items')

# --- MLE for Bradley-Terry scores ---
def bt_neg_log_likelihood(scores, pairs):
    scores = np.array(scores)
    loss = 0.0
    for w, l in pairs:
        loss -= np.log(sigmoid(scores[w] - scores[l]) + 1e-8)
    return loss

# Fix first score to 0 (identifiability)
def bt_nll_constrained(free_scores, pairs, N):
    scores = np.concatenate([[0.0], free_scores])
    return bt_neg_log_likelihood(scores, pairs)

res = minimize(
    bt_nll_constrained,
    x0=np.zeros(N_ITEMS - 1),
    args=(pairs, N_ITEMS),
    method='L-BFGS-B'
)
estimated_scores = np.concatenate([[0.0], res.x])

# Rank correlation
true_rank = np.argsort(-true_scores)
est_rank = np.argsort(-estimated_scores)
rank_corr = np.corrcoef(np.argsort(true_rank), np.argsort(est_rank))[0, 1]

print(f'True scores:      {true_scores}')
print(f'Estimated scores: {estimated_scores.round(3)}')
print(f'Rank correlation: {rank_corr:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
items = [f'Item {i}' for i in range(N_ITEMS)]

# Normalise for comparison
ts_norm = true_scores - true_scores.mean()
es_norm = estimated_scores - estimated_scores.mean()

x = np.arange(N_ITEMS)
axes[0].bar(x - 0.2, ts_norm, 0.4, label='True (latent)', color='#20808D')
axes[0].bar(x + 0.2, es_norm, 0.4, label='BT Estimated', color='#A84B2F', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(items)
axes[0].set_ylabel('Score (mean-centred)')
axes[0].set_title('Bradley-Terry: True vs Estimated Scores', fontweight='bold')
axes[0].legend()
axes[0].text(0.05, 0.95, f'Rank corr = {rank_corr:.3f}',
             transform=axes[0].transAxes, va='top', color='#1B474D', fontweight='bold')

# Preference probability heatmap
prob_matrix = np.zeros((N_ITEMS, N_ITEMS))
for i in range(N_ITEMS):
    for j in range(N_ITEMS):
        prob_matrix[i, j] = sigmoid(estimated_scores[i] - estimated_scores[j])

im = axes[1].imshow(prob_matrix, cmap='RdYlGn', vmin=0, vmax=1)
axes[1].set_xticks(range(N_ITEMS))
axes[1].set_yticks(range(N_ITEMS))
axes[1].set_xticklabels(items, rotation=45)
axes[1].set_yticklabels(items)
axes[1].set_title('P(row ≻ col) from BT Model', fontweight='bold')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.savefig('ch10_fig3_bradley_terry.png', dpi=120, bbox_inches='tight')
plt.show()

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§10.4 Reward Over-Optimisation (Goodhart's Law)</h2>
</div>

When we optimise the **proxy reward model** \(r_\phi\) instead of the true human reward \(r^*\), performance on the proxy initially tracks true quality — but eventually diverges. This is **Goodhart's Law** in RL:

> *When a measure becomes a target, it ceases to be a good measure.*

We simulate this by training a policy against a biased proxy reward (with added noise) and tracking correlation with the unobserved gold reward over optimisation steps.

In [ ]:
np.random.seed(7)

N_STEPS = 200
NOISE_LEVEL = 0.3        # How much proxy diverges from gold
EXPLOITATION_RATE = 0.05 # How aggressively policy exploits proxy

# Latent "quality" of responses in a continuous space
latent_quality = np.random.randn(1000)

# Gold reward: monotone function of quality
gold_reward = np.tanh(latent_quality)

# Proxy reward: noisy version
proxy_reward = gold_reward + NOISE_LEVEL * np.random.randn(1000)

# Simulate optimising the proxy: at each step, shift distribution
# toward higher proxy scores (gradient ascent in simplified latent space)
proxy_scores_over_time = []
gold_scores_over_time = []

# Policy = mixture weight toward high-proxy responses
weights = np.ones(1000) / 1000  # start uniform
for step in range(N_STEPS):
    # Weight more toward high proxy
    weights = weights * np.exp(EXPLOITATION_RATE * proxy_reward)
    weights /= weights.sum()

    proxy_scores_over_time.append(np.dot(weights, proxy_reward))
    gold_scores_over_time.append(np.dot(weights, gold_reward))

steps = np.arange(N_STEPS)

# Normalise to 0–1 range for comparison
proxy_arr = np.array(proxy_scores_over_time)
gold_arr = np.array(gold_scores_over_time)

proxy_norm = (proxy_arr - proxy_arr.min()) / (proxy_arr.max() - proxy_arr.min() + 1e-8)
gold_norm = (gold_arr - gold_arr[0]) / (gold_arr[0] + 1e-8)  # relative to start

print('Gold reward at step 0   :', gold_arr[0].round(4))
print('Gold reward at step 100 :', gold_arr[99].round(4))
print('Gold reward at step 199 :', gold_arr[-1].round(4))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: proxy vs gold over optimisation
ax1.plot(steps, proxy_arr, color='#20808D', linewidth=2, label='Proxy reward')
ax1.plot(steps, gold_arr, color='#A84B2F', linewidth=2, label='Gold reward', linestyle='--')
ax1.axvline(50, color='#7A7974', linestyle=':', alpha=0.7)
ax1.text(52, ax1.get_ylim()[0] + 0.05, 'Over-\noptimisation\nbegins', fontsize=8, color='#7A7974')
ax1.set_xlabel('Optimisation Steps')
ax1.set_ylabel('Expected Reward')
ax1.set_title('§10.4 — Reward Over-Optimisation\n(Goodhart's Law)', fontweight='bold')
ax1.legend()

# Right: scatter of gold vs proxy at different optimisation stages
stage_indices = [0, 50, 100, 199]
stage_colors = ['#BCE2E7', '#20808D', '#A84B2F', '#28251D']
stage_labels = ['Step 0 (init)', 'Step 50', 'Step 100', 'Step 199 (overopt)']

ax2.plot(proxy_arr, gold_arr, color='#D4D1CA', linewidth=1, zorder=1)
for idx, col, lbl in zip(stage_indices, stage_colors, stage_labels):
    ax2.scatter(proxy_arr[idx], gold_arr[idx], color=col, s=100, zorder=5, label=lbl)

ax2.set_xlabel('Proxy Reward')
ax2.set_ylabel('Gold Reward')
ax2.set_title('Proxy vs Gold Reward Trajectory', fontweight='bold')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('ch10_fig4_overoptimisation.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nKey insight: proxy reward keeps rising while gold reward peaks and falls.')
print('This is Goodhart\u2019s Law in action — the proxy is no longer a valid measure once we optimise it.')

<div style="background: #1C1B19; border-left: 4px solid #20808D; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">Summary & Key Takeaways</h2>
</div>

| Concept | Key Result |
|---------|------------|
| Potential-based shaping | Shapes reward with \(F = \gamma\Phi(s') - \Phi(s)\) — same optimal policy guaranteed |
| Reward model | A classifier's output probability serves as a scalar \(r_\phi \in [0,1]\) |
| Bradley-Terry | MLE over pairwise preferences recovers latent ranking (rank corr ≈ 1.0) |
| Over-optimisation | Proxy reward ↑, gold reward eventually ↓ — mitigation: KL penalty (Chapter 14) |

**Next:** Chapter 11 covers language model post-training: SFT, perplexity, and sampling strategies.